# Femora — Breast Ultrasound Classifier (ResNet50) + BUS-BRA

> **A separate notebook from v7.** It is v7's pipeline with a third hospital added (BUS-BRA: 1,875 scans, 4 scanners, Brazil).
> BUSI and BrEaST keep **exactly v7's train / validation / test split**, so the v7 test images are unchanged and the
> numbers are comparable. BUS-BRA is split **by patient**: ~60% train, ~10% validation, ~30% held out as a test set the
> model never trains on. v7 scored 77% sensitivity / 58% specificity / AUC 0.73 on all of BUS-BRA before seeing it.

**Datasets (combined)**
- **BUS-BRA**: 1,875 scans from 1,064 patients, 4 scanners, National Cancer Institute, Rio de Janeiro (Gómez-Flores et al.,
  Medical Physics 2024). Biopsy-proven benign / malignant only (no normal class). Cite the paper when using it.
- **BUSI**: 780 images from 600 women, Baheya Hospital, Cairo (Al-Dhabyani et al., 2020). Labels: normal / benign / malignant.
- **BrEaST-Lesions-USG**: 256 scans from 256 patients, every label confirmed by biopsy or follow-up (Pawłowska et al., 2024,
  The Cancer Imaging Archive, CC BY 4.0). A different hospital and different scanners, so it tests whether the model
  generalises beyond one clinic.

**Design decisions**
- **ResNet50 pretrained on ImageNet**, fine-tuned in two phases (classifier head first, then the whole network) with a
  class-weighted loss, because malignant and normal scans are the minority.
- **Grayscale, padded to a square**: scanners tint images differently and colour carries no information in B-mode
  ultrasound. Padding (instead of stretching) keeps lesion shape, which matters because irregular shape is a malignancy sign.
- **Near-duplicate images are grouped** by perceptual hash (BUSI contains several), so a copy of a test image can never be
  in the training set.
- **Two training recipes are compared by 5-fold cross-validation**: the baseline, and a recipe that weights both hospitals
  equally and adds scanner-style augmentation. The winner is chosen by a rule fixed in advance: the average malignant
  ROC-AUC across the two hospitals, on out-of-fold predictions. The test set is not used for any decision.
- **Screening threshold**: a screening tool should rarely miss cancer, so the "suspicious" threshold is tuned for ≥ 90%
  malignant sensitivity on ~870 out-of-fold predictions (a small validation split gives a noisy threshold).
- **Explainability**: Class Activation Maps (exact for a global-average-pool + linear head), checked against the
  radiologists' lesion masks.
- **Safety**: uploads that are not breast ultrasounds are rejected by a colour check and an "is this an ultrasound?"
  gate, evaluated on image types it never saw during training.
- **Every reported test number is computed with the exported ONNX model**, which is exactly what the Femora backend runs.

Output: normal / benign / malignant probabilities plus a heatmap. This is awareness only, not a diagnosis.

In [ ]:
%pip install -q onnx onnxruntime onnxscript

In [ ]:
import copy, glob, io, json, os, random, shutil, urllib.request, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageOps
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, recall_score, confusion_matrix,
                             classification_report, roc_curve)
warnings.filterwarnings("ignore")

# Paths can be overridden so the notebook can be smoke-tested locally before a Kaggle run
INPUT = os.environ.get("FEMORA_INPUT", "/kaggle/input")
WORK = os.environ.get("FEMORA_WORK", "/kaggle/working")
TMP = os.environ.get("FEMORA_TMP", "/tmp/femora")      # downloads go here so they don't bloat the notebook output
SMOKE = os.environ.get("FEMORA_SMOKE") == "1"          # tiny run: a few images, one epoch
OUT = f"{WORK}/model"
for d in (OUT, TMP):
    os.makedirs(d, exist_ok=True)

SEED = 42
CLASSES = ["normal", "benign", "malignant"]
SOURCES = ["BUSI", "BrEaST", "BUS-BRA"]
MAL = CLASSES.index("malignant")
IMG = 224
MEAN = np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]   # ImageNet statistics (pretrained backbone)
STD = np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]
COLOUR_LIMIT = 0.10       # max share of clearly coloured pixels in an upload
BATCH = 8 if SMOKE else 32
EPOCHS_HEAD, EPOCHS_FT, CV_FOLDS = (1, 1, 2) if SMOKE else (4, 26, 5)
NUM_WORKERS = 0 if os.name == "nt" else 4

def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def pick_device():
    if torch.cuda.is_available():
        try:  # fail over to CPU if this GPU architecture isn't in the installed torch build
            (torch.ones(2, 2, device="cuda") @ torch.ones(2, 2, device="cuda")).sum().item()
            return torch.device("cuda")
        except RuntimeError as e:
            print("GPU not usable:", e)
    return torch.device("cpu")

seed_everything()
DEVICE = pick_device()
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "", "torch", torch.__version__)

# ---- Preprocessing shared with backend/app.py (keep the two in sync)
def pad_square(img, fill=0):
    w, h = img.size
    s = max(w, h)
    canvas = Image.new(img.mode, (s, s), fill)
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas

def to_gray224(img):
    # any upload -> 224x224 uint8 grayscale, padded (not stretched) to a square
    return np.asarray(pad_square(ImageOps.exif_transpose(img).convert("L")).resize((IMG, IMG), Image.BILINEAR))

def normalize(gray224):
    x = gray224.astype(np.float32) / 255.0
    return ((np.repeat(x[None], 3, 0) - MEAN) / STD).astype(np.float32)

def colour_fraction(img):
    # share of clearly coloured pixels: B-mode ultrasound is grey (sometimes tinted), photos and Doppler are not
    a = np.asarray(img.convert("RGB").resize((128, 128)), np.int16)
    diff = np.maximum.reduce([abs(a[..., 0] - a[..., 1]), abs(a[..., 1] - a[..., 2]), abs(a[..., 0] - a[..., 2])])
    return float((diff > 30).mean())

def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def download(url, dest):
    if not os.path.exists(dest):
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f)
    return dest

In [ ]:
V7_SPLIT_CSV = '''file,source,label,group,split
normal (10).png,BUSI,normal,1,train
normal (100).png,BUSI,normal,2,train
normal (101).png,BUSI,normal,3,test
normal (102).png,BUSI,normal,4,train
normal (103).png,BUSI,normal,5,test
normal (104).png,BUSI,normal,6,val
normal (105).png,BUSI,normal,7,train
normal (106).png,BUSI,normal,8,test
normal (107).png,BUSI,normal,9,val
normal (108).png,BUSI,normal,10,train
normal (109).png,BUSI,normal,11,train
normal (11).png,BUSI,normal,12,train
normal (110).png,BUSI,normal,13,train
normal (111).png,BUSI,normal,14,train
normal (112).png,BUSI,normal,15,val
normal (113).png,BUSI,normal,16,test
normal (114).png,BUSI,normal,17,val
normal (115).png,BUSI,normal,18,train
normal (116).png,BUSI,normal,19,train
normal (117).png,BUSI,normal,20,train
normal (118).png,BUSI,normal,21,train
normal (119).png,BUSI,normal,22,train
normal (12).png,BUSI,normal,23,test
normal (120).png,BUSI,normal,24,val
normal (121).png,BUSI,normal,25,train
normal (122).png,BUSI,normal,26,train
normal (123).png,BUSI,normal,27,train
normal (124).png,BUSI,normal,53,val
normal (125).png,BUSI,normal,48,train
normal (126).png,BUSI,normal,54,train
normal (127).png,BUSI,normal,31,train
normal (128).png,BUSI,normal,32,train
normal (129).png,BUSI,normal,62,train
normal (13).png,BUSI,normal,78,test
normal (130).png,BUSI,normal,52,test
normal (131).png,BUSI,normal,36,val
normal (132).png,BUSI,normal,37,train
normal (133).png,BUSI,normal,38,test
normal (14).png,BUSI,normal,39,train
normal (15).png,BUSI,normal,40,train
normal (16).png,BUSI,normal,41,train
normal (17).png,BUSI,normal,42,train
normal (18).png,BUSI,normal,43,test
normal (19).png,BUSI,normal,44,train
normal (2).png,BUSI,normal,45,train
normal (20).png,BUSI,normal,46,train
normal (21).png,BUSI,normal,47,val
normal (22).png,BUSI,normal,48,train
normal (23).png,BUSI,normal,49,train
normal (24).png,BUSI,normal,50,train
normal (25).png,BUSI,normal,51,train
normal (26).png,BUSI,normal,52,test
normal (27).png,BUSI,normal,53,val
normal (28).png,BUSI,normal,54,train
normal (29).png,BUSI,normal,55,train
normal (3).png,BUSI,normal,56,train
normal (30).png,BUSI,normal,57,train
normal (31).png,BUSI,normal,58,train
normal (32).png,BUSI,normal,59,train
normal (33).png,BUSI,normal,60,train
normal (35).png,BUSI,normal,62,train
normal (36).png,BUSI,normal,63,train
normal (37).png,BUSI,normal,64,test
normal (38).png,BUSI,normal,65,val
normal (39).png,BUSI,normal,66,train
normal (4).png,BUSI,normal,67,train
normal (40).png,BUSI,normal,74,val
normal (41).png,BUSI,normal,91,train
normal (42).png,BUSI,normal,77,train
normal (43).png,BUSI,normal,71,train
normal (44).png,BUSI,normal,91,train
normal (45).png,BUSI,normal,77,train
normal (46).png,BUSI,normal,74,val
normal (47).png,BUSI,normal,91,train
normal (48).png,BUSI,normal,76,train
normal (49).png,BUSI,normal,77,train
normal (5).png,BUSI,normal,78,test
normal (50).png,BUSI,normal,79,test
normal (51).png,BUSI,normal,83,train
normal (52).png,BUSI,normal,81,train
normal (53).png,BUSI,normal,82,train
normal (54).png,BUSI,normal,83,train
normal (55).png,BUSI,normal,84,train
normal (56).png,BUSI,normal,85,val
normal (57).png,BUSI,normal,86,train
normal (58).png,BUSI,normal,99,train
normal (59).png,BUSI,normal,88,test
normal (6).png,BUSI,normal,89,train
normal (60).png,BUSI,normal,99,train
normal (61).png,BUSI,normal,91,train
normal (62).png,BUSI,normal,77,train
normal (63).png,BUSI,normal,93,train
normal (64).png,BUSI,normal,94,train
normal (65).png,BUSI,normal,95,train
normal (66).png,BUSI,normal,96,train
normal (67).png,BUSI,normal,97,train
normal (68).png,BUSI,normal,98,test
normal (69).png,BUSI,normal,99,train
normal (7).png,BUSI,normal,100,val
normal (70).png,BUSI,normal,101,train
normal (71).png,BUSI,normal,102,train
normal (72).png,BUSI,normal,103,train
normal (73).png,BUSI,normal,104,test
normal (74).png,BUSI,normal,105,train
normal (75).png,BUSI,normal,106,train
normal (76).png,BUSI,normal,107,val
normal (77).png,BUSI,normal,108,train
normal (78).png,BUSI,normal,109,train
normal (79).png,BUSI,normal,110,train
normal (8).png,BUSI,normal,111,train
normal (80).png,BUSI,normal,112,train
normal (81).png,BUSI,normal,113,test
normal (82).png,BUSI,normal,114,train
normal (83).png,BUSI,normal,115,train
normal (84).png,BUSI,normal,116,train
normal (85).png,BUSI,normal,117,val
normal (86).png,BUSI,normal,118,test
normal (87).png,BUSI,normal,119,train
normal (88).png,BUSI,normal,120,train
normal (89).png,BUSI,normal,121,train
normal (9).png,BUSI,normal,122,test
normal (90).png,BUSI,normal,123,train
normal (91).png,BUSI,normal,124,val
normal (92).png,BUSI,normal,125,val
normal (93).png,BUSI,normal,126,train
normal (94).png,BUSI,normal,127,train
normal (95).png,BUSI,normal,128,train
normal (96).png,BUSI,normal,129,train
normal (97).png,BUSI,normal,131,test
normal (98).png,BUSI,normal,131,test
normal (99).png,BUSI,normal,132,val
benign (1).png,BUSI,benign,133,train
benign (10).png,BUSI,benign,386,train
benign (100).png,BUSI,benign,135,train
benign (101).png,BUSI,benign,136,train
benign (102).png,BUSI,benign,137,train
benign (103).png,BUSI,benign,138,train
benign (104).png,BUSI,benign,139,train
benign (105).png,BUSI,benign,196,train
benign (106).png,BUSI,benign,141,test
benign (107).png,BUSI,benign,142,train
benign (108).png,BUSI,benign,143,test
benign (109).png,BUSI,benign,144,train
benign (11).png,BUSI,benign,145,train
benign (110).png,BUSI,benign,146,val
benign (111).png,BUSI,benign,147,train
benign (112).png,BUSI,benign,148,train
benign (113).png,BUSI,benign,149,train
benign (114).png,BUSI,benign,150,train
benign (115).png,BUSI,benign,151,test
benign (116).png,BUSI,benign,152,val
benign (117).png,BUSI,benign,153,train
benign (118).png,BUSI,benign,154,train
benign (119).png,BUSI,benign,155,train
benign (12).png,BUSI,benign,156,val
benign (120).png,BUSI,benign,157,train
benign (121).png,BUSI,benign,158,test
benign (122).png,BUSI,benign,159,train
benign (123).png,BUSI,benign,160,train
benign (124).png,BUSI,benign,161,val
benign (125).png,BUSI,benign,162,train
benign (126).png,BUSI,benign,163,train
benign (127).png,BUSI,benign,164,train
benign (128).png,BUSI,benign,165,train
benign (129).png,BUSI,benign,166,train
benign (13).png,BUSI,benign,390,train
benign (130).png,BUSI,benign,168,test
benign (132).png,BUSI,benign,170,val
benign (133).png,BUSI,benign,171,train
benign (134).png,BUSI,benign,172,train
benign (135).png,BUSI,benign,173,train
benign (136).png,BUSI,benign,174,train
benign (137).png,BUSI,benign,175,train
benign (138).png,BUSI,benign,527,val
benign (139).png,BUSI,benign,532,train
benign (14).png,BUSI,benign,178,train
benign (140).png,BUSI,benign,179,test
benign (141).png,BUSI,benign,180,train
benign (142).png,BUSI,benign,181,train
benign (143).png,BUSI,benign,182,val
benign (144).png,BUSI,benign,183,train
benign (145).png,BUSI,benign,184,train
benign (146).png,BUSI,benign,185,train
benign (147).png,BUSI,benign,186,train
benign (148).png,BUSI,benign,187,train
benign (149).png,BUSI,benign,188,val
benign (15).png,BUSI,benign,392,val
benign (150).png,BUSI,benign,190,train
benign (151).png,BUSI,benign,191,train
benign (152).png,BUSI,benign,222,train
benign (153).png,BUSI,benign,300,val
benign (154).png,BUSI,benign,194,test
benign (155).png,BUSI,benign,195,train
benign (156).png,BUSI,benign,196,train
benign (157).png,BUSI,benign,532,train
benign (158).png,BUSI,benign,198,train
benign (159).png,BUSI,benign,199,train
benign (16).png,BUSI,benign,200,test
benign (160).png,BUSI,benign,201,train
benign (161).png,BUSI,benign,202,val
benign (162).png,BUSI,benign,203,train
benign (163).png,BUSI,benign,555,train
benign (165).png,BUSI,benign,206,train
benign (166).png,BUSI,benign,207,train
benign (167).png,BUSI,benign,208,train
benign (168).png,BUSI,benign,209,train
benign (169).png,BUSI,benign,210,test
benign (17).png,BUSI,benign,211,train
benign (170).png,BUSI,benign,212,val
benign (171).png,BUSI,benign,213,train
benign (172).png,BUSI,benign,214,test
benign (173).png,BUSI,benign,215,val
benign (174).png,BUSI,benign,216,train
benign (175).png,BUSI,benign,217,train
benign (176).png,BUSI,benign,218,train
benign (177).png,BUSI,benign,411,test
benign (178).png,BUSI,benign,220,train
benign (179).png,BUSI,benign,221,val
benign (18).png,BUSI,benign,222,train
benign (180).png,BUSI,benign,223,train
benign (181).png,BUSI,benign,224,test
benign (182).png,BUSI,benign,225,train
benign (183).png,BUSI,benign,226,train
benign (184).png,BUSI,benign,227,train
benign (185).png,BUSI,benign,228,train
benign (186).png,BUSI,benign,229,train
benign (187).png,BUSI,benign,230,train
benign (188).png,BUSI,benign,231,train
benign (189).png,BUSI,benign,232,train
benign (19).png,BUSI,benign,233,train
benign (190).png,BUSI,benign,234,train
benign (191).png,BUSI,benign,235,test
benign (192).png,BUSI,benign,236,val
benign (193).png,BUSI,benign,237,test
benign (194).png,BUSI,benign,238,train
benign (195).png,BUSI,benign,239,train
benign (196).png,BUSI,benign,240,train
benign (197).png,BUSI,benign,241,train
benign (198).png,BUSI,benign,242,train
benign (199).png,BUSI,benign,243,test
benign (2).png,BUSI,benign,244,train
benign (20).png,BUSI,benign,245,val
benign (200).png,BUSI,benign,246,train
benign (201).png,BUSI,benign,247,train
benign (202).png,BUSI,benign,358,train
benign (203).png,BUSI,benign,343,train
benign (204).png,BUSI,benign,336,train
benign (205).png,BUSI,benign,251,test
benign (206).png,BUSI,benign,252,train
benign (207).png,BUSI,benign,253,val
benign (208).png,BUSI,benign,254,train
benign (209).png,BUSI,benign,255,train
benign (21).png,BUSI,benign,256,train
benign (210).png,BUSI,benign,353,test
benign (211).png,BUSI,benign,258,val
benign (212).png,BUSI,benign,259,train
benign (213).png,BUSI,benign,260,train
benign (214).png,BUSI,benign,335,train
benign (215).png,BUSI,benign,323,train
benign (216).png,BUSI,benign,317,train
benign (217).png,BUSI,benign,320,val
benign (218).png,BUSI,benign,351,train
benign (219).png,BUSI,benign,266,test
benign (22).png,BUSI,benign,267,train
benign (220).png,BUSI,benign,324,train
benign (221).png,BUSI,benign,269,train
benign (222).png,BUSI,benign,270,val
benign (223).png,BUSI,benign,271,train
benign (224).png,BUSI,benign,272,train
benign (225).png,BUSI,benign,348,train
benign (226).png,BUSI,benign,331,val
benign (227).png,BUSI,benign,301,train
benign (228).png,BUSI,benign,363,train
benign (229).png,BUSI,benign,277,train
benign (23).png,BUSI,benign,278,train
benign (230).png,BUSI,benign,279,test
benign (231).png,BUSI,benign,350,train
benign (232).png,BUSI,benign,341,val
benign (233).png,BUSI,benign,354,val
benign (234).png,BUSI,benign,313,train
benign (235).png,BUSI,benign,349,train
benign (236).png,BUSI,benign,285,train
benign (237).png,BUSI,benign,286,val
benign (238).png,BUSI,benign,357,train
benign (239).png,BUSI,benign,288,train
benign (24).png,BUSI,benign,289,test
benign (240).png,BUSI,benign,330,test
benign (241).png,BUSI,benign,291,train
benign (242).png,BUSI,benign,342,train
benign (243).png,BUSI,benign,293,train
benign (244).png,BUSI,benign,294,train
benign (245).png,BUSI,benign,295,train
benign (246).png,BUSI,benign,296,train
benign (247).png,BUSI,benign,297,train
benign (248).png,BUSI,benign,298,train
benign (249).png,BUSI,benign,336,train
benign (25).png,BUSI,benign,300,val
benign (250).png,BUSI,benign,301,train
benign (251).png,BUSI,benign,302,train
benign (252).png,BUSI,benign,303,train
benign (253).png,BUSI,benign,304,train
benign (254).png,BUSI,benign,305,train
benign (255).png,BUSI,benign,330,test
benign (256).png,BUSI,benign,307,test
benign (257).png,BUSI,benign,308,val
benign (258).png,BUSI,benign,309,train
benign (259).png,BUSI,benign,334,train
benign (26).png,BUSI,benign,311,train
benign (260).png,BUSI,benign,312,train
benign (261).png,BUSI,benign,313,train
benign (262).png,BUSI,benign,314,train
benign (263).png,BUSI,benign,331,val
benign (264).png,BUSI,benign,316,train
benign (265).png,BUSI,benign,317,train
benign (266).png,BUSI,benign,332,train
benign (267).png,BUSI,benign,319,test
benign (268).png,BUSI,benign,320,val
benign (27).png,BUSI,benign,322,train
benign (270).png,BUSI,benign,323,train
benign (271).png,BUSI,benign,324,train
benign (272).png,BUSI,benign,325,train
benign (273).png,BUSI,benign,343,train
benign (274).png,BUSI,benign,345,test
benign (275).png,BUSI,benign,352,train
benign (276).png,BUSI,benign,341,val
benign (277).png,BUSI,benign,330,test
benign (278).png,BUSI,benign,331,val
benign (279).png,BUSI,benign,332,train
benign (28).png,BUSI,benign,333,train
benign (280).png,BUSI,benign,334,train
benign (281).png,BUSI,benign,335,train
benign (282).png,BUSI,benign,336,train
benign (283).png,BUSI,benign,337,train
benign (284).png,BUSI,benign,339,test
benign (285).png,BUSI,benign,339,test
benign (286).png,BUSI,benign,340,train
benign (287).png,BUSI,benign,341,val
benign (288).png,BUSI,benign,342,train
benign (289).png,BUSI,benign,343,train
benign (29).png,BUSI,benign,344,train
benign (290).png,BUSI,benign,345,test
benign (291).png,BUSI,benign,346,val
benign (292).png,BUSI,benign,348,train
benign (293).png,BUSI,benign,348,train
benign (294).png,BUSI,benign,349,train
benign (295).png,BUSI,benign,350,train
benign (296).png,BUSI,benign,351,train
benign (297).png,BUSI,benign,352,train
benign (298).png,BUSI,benign,353,test
benign (299).png,BUSI,benign,354,val
benign (3).png,BUSI,benign,379,test
benign (30).png,BUSI,benign,356,train
benign (300).png,BUSI,benign,357,train
benign (301).png,BUSI,benign,358,train
benign (302).png,BUSI,benign,359,val
benign (303).png,BUSI,benign,360,train
benign (304).png,BUSI,benign,361,train
benign (305).png,BUSI,benign,362,train
benign (306).png,BUSI,benign,363,train
benign (307).png,BUSI,benign,488,train
benign (308).png,BUSI,benign,365,train
benign (309).png,BUSI,benign,492,train
benign (31).png,BUSI,benign,367,train
benign (310).png,BUSI,benign,368,test
benign (311).png,BUSI,benign,369,train
benign (312).png,BUSI,benign,491,val
benign (313).png,BUSI,benign,371,train
benign (314).png,BUSI,benign,372,train
benign (315).png,BUSI,benign,373,test
benign (316).png,BUSI,benign,496,test
benign (317).png,BUSI,benign,375,val
benign (318).png,BUSI,benign,376,train
benign (319).png,BUSI,benign,377,train
benign (32).png,BUSI,benign,378,test
benign (320).png,BUSI,benign,379,test
benign (321).png,BUSI,benign,466,train
benign (322).png,BUSI,benign,515,train
benign (323).png,BUSI,benign,382,train
benign (324).png,BUSI,benign,537,train
benign (325).png,BUSI,benign,384,train
benign (326).png,BUSI,benign,559,train
benign (327).png,BUSI,benign,386,train
benign (328).png,BUSI,benign,387,val
benign (329).png,BUSI,benign,388,train
benign (33).png,BUSI,benign,389,test
benign (330).png,BUSI,benign,390,train
benign (331).png,BUSI,benign,391,train
benign (332).png,BUSI,benign,392,val
benign (333).png,BUSI,benign,393,train
benign (334).png,BUSI,benign,394,train
benign (335).png,BUSI,benign,395,test
benign (336).png,BUSI,benign,396,val
benign (337).png,BUSI,benign,397,train
benign (338).png,BUSI,benign,398,train
benign (339).png,BUSI,benign,399,train
benign (34).png,BUSI,benign,400,train
benign (340).png,BUSI,benign,401,train
benign (341).png,BUSI,benign,402,test
benign (342).png,BUSI,benign,403,val
benign (343).png,BUSI,benign,404,train
benign (344).png,BUSI,benign,405,train
benign (345).png,BUSI,benign,406,val
benign (346).png,BUSI,benign,407,train
benign (347).png,BUSI,benign,408,train
benign (348).png,BUSI,benign,409,train
benign (349).png,BUSI,benign,410,train
benign (35).png,BUSI,benign,411,test
benign (350).png,BUSI,benign,412,train
benign (351).png,BUSI,benign,413,train
benign (352).png,BUSI,benign,414,train
benign (353).png,BUSI,benign,415,val
benign (354).png,BUSI,benign,416,train
benign (355).png,BUSI,benign,417,test
benign (356).png,BUSI,benign,418,train
benign (357).png,BUSI,benign,419,val
benign (358).png,BUSI,benign,420,train
benign (359).png,BUSI,benign,421,train
benign (36).png,BUSI,benign,422,train
benign (360).png,BUSI,benign,423,train
benign (361).png,BUSI,benign,424,train
benign (362).png,BUSI,benign,425,test
benign (363).png,BUSI,benign,426,train
benign (364).png,BUSI,benign,427,val
benign (365).png,BUSI,benign,428,train
benign (366).png,BUSI,benign,429,train
benign (367).png,BUSI,benign,430,train
benign (368).png,BUSI,benign,431,train
benign (369).png,BUSI,benign,432,train
benign (37).png,BUSI,benign,433,train
benign (370).png,BUSI,benign,434,test
benign (371).png,BUSI,benign,435,val
benign (372).png,BUSI,benign,436,train
benign (373).png,BUSI,benign,437,train
benign (374).png,BUSI,benign,438,train
benign (375).png,BUSI,benign,439,train
benign (376).png,BUSI,benign,440,test
benign (377).png,BUSI,benign,441,train
benign (378).png,BUSI,benign,442,train
benign (379).png,BUSI,benign,443,train
benign (38).png,BUSI,benign,444,val
benign (380).png,BUSI,benign,445,train
benign (381).png,BUSI,benign,446,train
benign (382).png,BUSI,benign,447,train
benign (383).png,BUSI,benign,448,val
benign (384).png,BUSI,benign,449,train
benign (385).png,BUSI,benign,450,test
benign (386).png,BUSI,benign,451,train
benign (387).png,BUSI,benign,452,train
benign (388).png,BUSI,benign,453,train
benign (389).png,BUSI,benign,454,train
benign (39).png,BUSI,benign,455,train
benign (390).png,BUSI,benign,456,train
benign (391).png,BUSI,benign,457,train
benign (392).png,BUSI,benign,458,train
benign (393).png,BUSI,benign,459,test
benign (394).png,BUSI,benign,460,train
benign (395).png,BUSI,benign,480,test
benign (396).png,BUSI,benign,462,train
benign (397).png,BUSI,benign,463,train
benign (398).png,BUSI,benign,464,val
benign (4).png,BUSI,benign,466,train
benign (40).png,BUSI,benign,467,train
benign (400).png,BUSI,benign,481,test
benign (401).png,BUSI,benign,469,train
benign (402).png,BUSI,benign,470,train
benign (403).png,BUSI,benign,471,test
benign (404).png,BUSI,benign,472,val
benign (405).png,BUSI,benign,473,train
benign (406).png,BUSI,benign,474,train
benign (407).png,BUSI,benign,475,train
benign (408).png,BUSI,benign,476,train
benign (409).png,BUSI,benign,477,train
benign (41).png,BUSI,benign,478,train
benign (410).png,BUSI,benign,479,train
benign (411).png,BUSI,benign,480,test
benign (412).png,BUSI,benign,481,test
benign (413).png,BUSI,benign,482,val
benign (414).png,BUSI,benign,483,train
benign (415).png,BUSI,benign,484,train
benign (416).png,BUSI,benign,485,train
benign (417).png,BUSI,benign,486,train
benign (418).png,BUSI,benign,487,val
benign (419).png,BUSI,benign,488,train
benign (420).png,BUSI,benign,490,test
benign (421).png,BUSI,benign,491,val
benign (422).png,BUSI,benign,492,train
benign (423).png,BUSI,benign,493,train
benign (424).png,BUSI,benign,494,train
benign (425).png,BUSI,benign,495,train
benign (426).png,BUSI,benign,496,test
benign (427).png,BUSI,benign,497,train
benign (428).png,BUSI,benign,498,train
benign (429).png,BUSI,benign,499,val
benign (43).png,BUSI,benign,500,train
benign (430).png,BUSI,benign,501,val
benign (431).png,BUSI,benign,502,train
benign (432).png,BUSI,benign,503,train
benign (434).png,BUSI,benign,505,train
benign (435).png,BUSI,benign,506,train
benign (436).png,BUSI,benign,507,train
benign (44).png,BUSI,benign,509,test
benign (45).png,BUSI,benign,510,train
benign (46).png,BUSI,benign,511,val
benign (47).png,BUSI,benign,512,train
benign (48).png,BUSI,benign,513,train
benign (49).png,BUSI,benign,514,train
benign (5).png,BUSI,benign,515,train
benign (50).png,BUSI,benign,516,test
benign (51).png,BUSI,benign,517,train
benign (52).png,BUSI,benign,518,train
benign (53).png,BUSI,benign,519,train
benign (54).png,BUSI,benign,520,train
benign (55).png,BUSI,benign,521,val
benign (56).png,BUSI,benign,522,train
benign (57).png,BUSI,benign,523,train
benign (58).png,BUSI,benign,524,train
benign (59).png,BUSI,benign,525,test
benign (6).png,BUSI,benign,526,train
benign (60).png,BUSI,benign,527,val
benign (61).png,BUSI,benign,528,train
benign (62).png,BUSI,benign,529,train
benign (63).png,BUSI,benign,530,train
benign (64).png,BUSI,benign,531,val
benign (65).png,BUSI,benign,532,train
benign (66).png,BUSI,benign,533,train
benign (67).png,BUSI,benign,534,test
benign (68).png,BUSI,benign,535,train
benign (69).png,BUSI,benign,536,train
benign (7).png,BUSI,benign,537,train
benign (70).png,BUSI,benign,538,train
benign (71).png,BUSI,benign,539,train
benign (72).png,BUSI,benign,540,test
benign (73).png,BUSI,benign,541,train
benign (74).png,BUSI,benign,542,train
benign (75).png,BUSI,benign,543,val
benign (76).png,BUSI,benign,544,train
benign (77).png,BUSI,benign,545,train
benign (78).png,BUSI,benign,546,val
benign (79).png,BUSI,benign,547,train
benign (8).png,BUSI,benign,548,train
benign (80).png,BUSI,benign,549,train
benign (81).png,BUSI,benign,550,test
benign (82).png,BUSI,benign,551,train
benign (83).png,BUSI,benign,552,train
benign (84).png,BUSI,benign,553,train
benign (86).png,BUSI,benign,555,train
benign (87).png,BUSI,benign,556,test
benign (88).png,BUSI,benign,557,val
benign (89).png,BUSI,benign,558,train
benign (9).png,BUSI,benign,559,train
benign (90).png,BUSI,benign,560,train
benign (91).png,BUSI,benign,561,train
benign (92).png,BUSI,benign,562,train
benign (93).png,BUSI,benign,563,train
benign (94).png,BUSI,benign,564,test
benign (95).png,BUSI,benign,565,val
benign (96).png,BUSI,benign,566,train
benign (97).png,BUSI,benign,567,train
benign (98).png,BUSI,benign,568,train
benign (99).png,BUSI,benign,569,train
malignant (1).png,BUSI,malignant,570,train
malignant (10).png,BUSI,malignant,571,val
malignant (100).png,BUSI,malignant,572,train
malignant (101).png,BUSI,malignant,573,train
malignant (102).png,BUSI,malignant,574,test
malignant (103).png,BUSI,malignant,575,train
malignant (104).png,BUSI,malignant,576,test
malignant (105).png,BUSI,malignant,577,train
malignant (106).png,BUSI,malignant,578,train
malignant (107).png,BUSI,malignant,579,train
malignant (108).png,BUSI,malignant,580,test
malignant (109).png,BUSI,malignant,581,test
malignant (11).png,BUSI,malignant,582,val
malignant (110).png,BUSI,malignant,583,train
malignant (111).png,BUSI,malignant,584,val
malignant (112).png,BUSI,malignant,585,train
malignant (113).png,BUSI,malignant,586,train
malignant (114).png,BUSI,malignant,587,train
malignant (115).png,BUSI,malignant,588,val
malignant (116).png,BUSI,malignant,659,train
malignant (117).png,BUSI,malignant,590,train
malignant (118).png,BUSI,malignant,591,train
malignant (119).png,BUSI,malignant,592,train
malignant (12).png,BUSI,malignant,593,train
malignant (120).png,BUSI,malignant,594,test
malignant (121).png,BUSI,malignant,595,train
malignant (122).png,BUSI,malignant,596,val
malignant (123).png,BUSI,malignant,597,train
malignant (124).png,BUSI,malignant,598,train
malignant (125).png,BUSI,malignant,599,train
malignant (126).png,BUSI,malignant,600,train
malignant (127).png,BUSI,malignant,601,train
malignant (128).png,BUSI,malignant,602,train
malignant (129).png,BUSI,malignant,603,train
malignant (13).png,BUSI,malignant,604,test
malignant (130).png,BUSI,malignant,605,train
malignant (131).png,BUSI,malignant,606,val
malignant (132).png,BUSI,malignant,607,train
malignant (133).png,BUSI,malignant,608,test
malignant (134).png,BUSI,malignant,609,train
malignant (135).png,BUSI,malignant,610,train
malignant (136).png,BUSI,malignant,611,train
malignant (137).png,BUSI,malignant,612,train
malignant (138).png,BUSI,malignant,613,train
malignant (139).png,BUSI,malignant,614,train
malignant (14).png,BUSI,malignant,615,val
malignant (140).png,BUSI,malignant,616,train
malignant (141).png,BUSI,malignant,617,train
malignant (142).png,BUSI,malignant,618,test
malignant (143).png,BUSI,malignant,619,train
malignant (144).png,BUSI,malignant,620,train
malignant (146).png,BUSI,malignant,622,val
malignant (147).png,BUSI,malignant,623,train
malignant (148).png,BUSI,malignant,624,train
malignant (149).png,BUSI,malignant,625,test
malignant (15).png,BUSI,malignant,626,train
malignant (150).png,BUSI,malignant,627,train
malignant (151).png,BUSI,malignant,628,train
malignant (152).png,BUSI,malignant,629,train
malignant (153).png,BUSI,malignant,630,test
malignant (154).png,BUSI,malignant,631,train
malignant (155).png,BUSI,malignant,632,val
malignant (156).png,BUSI,malignant,633,train
malignant (157).png,BUSI,malignant,634,train
malignant (158).png,BUSI,malignant,635,train
malignant (159).png,BUSI,malignant,636,train
malignant (16).png,BUSI,malignant,637,train
malignant (160).png,BUSI,malignant,638,train
malignant (161).png,BUSI,malignant,639,train
malignant (162).png,BUSI,malignant,640,test
malignant (163).png,BUSI,malignant,641,train
malignant (164).png,BUSI,malignant,642,val
malignant (165).png,BUSI,malignant,643,train
malignant (166).png,BUSI,malignant,644,train
malignant (167).png,BUSI,malignant,645,val
malignant (168).png,BUSI,malignant,646,train
malignant (169).png,BUSI,malignant,647,train
malignant (17).png,BUSI,malignant,648,test
malignant (170).png,BUSI,malignant,649,val
malignant (171).png,BUSI,malignant,650,train
malignant (172).png,BUSI,malignant,651,train
malignant (173).png,BUSI,malignant,652,train
malignant (174).png,BUSI,malignant,653,test
malignant (175).png,BUSI,malignant,654,train
malignant (176).png,BUSI,malignant,655,train
malignant (177).png,BUSI,malignant,656,train
malignant (178).png,BUSI,malignant,657,val
malignant (179).png,BUSI,malignant,658,train
malignant (18).png,BUSI,malignant,659,train
malignant (180).png,BUSI,malignant,660,train
malignant (181).png,BUSI,malignant,661,train
malignant (182).png,BUSI,malignant,662,test
malignant (183).png,BUSI,malignant,663,train
malignant (184).png,BUSI,malignant,664,train
malignant (185).png,BUSI,malignant,665,train
malignant (186).png,BUSI,malignant,666,train
malignant (187).png,BUSI,malignant,667,train
malignant (188).png,BUSI,malignant,668,train
malignant (189).png,BUSI,malignant,669,test
malignant (19).png,BUSI,malignant,670,val
malignant (190).png,BUSI,malignant,671,train
malignant (191).png,BUSI,malignant,672,train
malignant (192).png,BUSI,malignant,673,test
malignant (193).png,BUSI,malignant,674,val
malignant (194).png,BUSI,malignant,675,train
malignant (195).png,BUSI,malignant,676,train
malignant (196).png,BUSI,malignant,677,test
malignant (197).png,BUSI,malignant,678,train
malignant (198).png,BUSI,malignant,679,train
malignant (199).png,BUSI,malignant,680,val
malignant (2).png,BUSI,malignant,681,train
malignant (20).png,BUSI,malignant,682,train
malignant (200).png,BUSI,malignant,683,train
malignant (201).png,BUSI,malignant,684,train
malignant (202).png,BUSI,malignant,685,train
malignant (203).png,BUSI,malignant,686,train
malignant (204).png,BUSI,malignant,687,test
malignant (205).png,BUSI,malignant,688,val
malignant (206).png,BUSI,malignant,689,train
malignant (207).png,BUSI,malignant,690,train
malignant (208).png,BUSI,malignant,691,train
malignant (209).png,BUSI,malignant,692,val
malignant (21).png,BUSI,malignant,693,train
malignant (210).png,BUSI,malignant,694,train
malignant (22).png,BUSI,malignant,695,train
malignant (23).png,BUSI,malignant,696,train
malignant (24).png,BUSI,malignant,697,test
malignant (25).png,BUSI,malignant,698,train
malignant (26).png,BUSI,malignant,699,val
malignant (27).png,BUSI,malignant,700,train
malignant (28).png,BUSI,malignant,701,train
malignant (29).png,BUSI,malignant,702,train
malignant (3).png,BUSI,malignant,703,train
malignant (30).png,BUSI,malignant,704,train
malignant (31).png,BUSI,malignant,705,train
malignant (32).png,BUSI,malignant,706,train
malignant (33).png,BUSI,malignant,707,train
malignant (34).png,BUSI,malignant,708,test
malignant (35).png,BUSI,malignant,709,train
malignant (36).png,BUSI,malignant,710,test
malignant (37).png,BUSI,malignant,711,train
malignant (38).png,BUSI,malignant,712,val
malignant (39).png,BUSI,malignant,713,train
malignant (4).png,BUSI,malignant,714,train
malignant (40).png,BUSI,malignant,715,train
malignant (41).png,BUSI,malignant,716,test
malignant (42).png,BUSI,malignant,717,val
malignant (43).png,BUSI,malignant,718,train
malignant (44).png,BUSI,malignant,719,train
malignant (45).png,BUSI,malignant,720,train
malignant (46).png,BUSI,malignant,721,train
malignant (47).png,BUSI,malignant,722,train
malignant (48).png,BUSI,malignant,723,test
malignant (49).png,BUSI,malignant,724,val
malignant (5).png,BUSI,malignant,725,train
malignant (50).png,BUSI,malignant,726,train
malignant (53).png,BUSI,malignant,729,test
malignant (54).png,BUSI,malignant,730,train
malignant (55).png,BUSI,malignant,731,train
malignant (56).png,BUSI,malignant,732,val
malignant (57).png,BUSI,malignant,733,train
malignant (58).png,BUSI,malignant,734,test
malignant (59).png,BUSI,malignant,735,train
malignant (6).png,BUSI,malignant,736,train
malignant (60).png,BUSI,malignant,737,train
malignant (61).png,BUSI,malignant,738,train
malignant (62).png,BUSI,malignant,739,val
malignant (63).png,BUSI,malignant,740,train
malignant (64).png,BUSI,malignant,741,train
malignant (65).png,BUSI,malignant,742,train
malignant (66).png,BUSI,malignant,743,test
malignant (67).png,BUSI,malignant,744,train
malignant (68).png,BUSI,malignant,745,val
malignant (69).png,BUSI,malignant,746,train
malignant (7).png,BUSI,malignant,747,train
malignant (70).png,BUSI,malignant,748,train
malignant (71).png,BUSI,malignant,749,test
malignant (72).png,BUSI,malignant,750,train
malignant (73).png,BUSI,malignant,751,train
malignant (74).png,BUSI,malignant,752,train
malignant (75).png,BUSI,malignant,753,train
malignant (76).png,BUSI,malignant,754,train
malignant (77).png,BUSI,malignant,755,train
malignant (78).png,BUSI,malignant,756,train
malignant (79).png,BUSI,malignant,757,train
malignant (8).png,BUSI,malignant,758,val
malignant (80).png,BUSI,malignant,759,train
malignant (81).png,BUSI,malignant,760,test
malignant (82).png,BUSI,malignant,761,train
malignant (83).png,BUSI,malignant,762,val
malignant (84).png,BUSI,malignant,763,train
malignant (85).png,BUSI,malignant,764,train
malignant (86).png,BUSI,malignant,765,train
malignant (87).png,BUSI,malignant,766,train
malignant (88).png,BUSI,malignant,767,test
malignant (89).png,BUSI,malignant,768,test
malignant (9).png,BUSI,malignant,769,train
malignant (90).png,BUSI,malignant,770,train
malignant (91).png,BUSI,malignant,771,val
malignant (92).png,BUSI,malignant,772,train
malignant (94).png,BUSI,malignant,774,train
malignant (95).png,BUSI,malignant,775,train
malignant (96).png,BUSI,malignant,776,test
malignant (97).png,BUSI,malignant,777,val
malignant (98).png,BUSI,malignant,778,train
malignant (99).png,BUSI,malignant,779,train
case001.png,BrEaST,benign,780,train
case002.png,BrEaST,benign,781,train
case003.png,BrEaST,benign,782,val
case004.png,BrEaST,benign,783,train
case005.png,BrEaST,malignant,784,train
case006.png,BrEaST,benign,785,train
case007.png,BrEaST,malignant,786,val
case008.png,BrEaST,malignant,787,test
case009.png,BrEaST,benign,788,train
case010.png,BrEaST,malignant,789,train
case011.png,BrEaST,malignant,790,train
case012.png,BrEaST,malignant,791,train
case013.png,BrEaST,malignant,792,train
case014.png,BrEaST,benign,793,val
case015.png,BrEaST,benign,794,test
case016.png,BrEaST,benign,795,train
case017.png,BrEaST,benign,796,val
case018.png,BrEaST,benign,797,train
case019.png,BrEaST,benign,798,train
case020.png,BrEaST,benign,799,train
case021.png,BrEaST,benign,800,train
case022.png,BrEaST,benign,801,test
case023.png,BrEaST,benign,802,train
case024.png,BrEaST,benign,803,train
case025.png,BrEaST,benign,804,train
case026.png,BrEaST,malignant,805,train
case027.png,BrEaST,malignant,806,train
case028.png,BrEaST,benign,807,train
case029.png,BrEaST,benign,808,val
case030.png,BrEaST,benign,809,train
case031.png,BrEaST,benign,810,train
case032.png,BrEaST,malignant,811,train
case033.png,BrEaST,benign,812,train
case034.png,BrEaST,malignant,813,test
case035.png,BrEaST,malignant,814,train
case036.png,BrEaST,benign,815,val
case037.png,BrEaST,benign,816,train
case038.png,BrEaST,benign,817,train
case039.png,BrEaST,benign,818,train
case040.png,BrEaST,benign,819,train
case041.png,BrEaST,benign,820,train
case042.png,BrEaST,benign,821,train
case043.png,BrEaST,benign,822,train
case044.png,BrEaST,benign,823,train
case045.png,BrEaST,normal,824,test
case046.png,BrEaST,malignant,825,val
case047.png,BrEaST,benign,826,train
case048.png,BrEaST,benign,827,train
case049.png,BrEaST,benign,828,train
case050.png,BrEaST,malignant,829,train
case051.png,BrEaST,benign,830,test
case052.png,BrEaST,benign,831,train
case053.png,BrEaST,malignant,832,test
case054.png,BrEaST,benign,833,train
case055.png,BrEaST,malignant,834,test
case056.png,BrEaST,benign,835,train
case057.png,BrEaST,benign,836,val
case058.png,BrEaST,malignant,837,train
case059.png,BrEaST,benign,838,train
case060.png,BrEaST,malignant,839,test
case061.png,BrEaST,normal,840,val
case062.png,BrEaST,benign,841,train
case063.png,BrEaST,malignant,842,train
case064.png,BrEaST,benign,843,val
case065.png,BrEaST,benign,844,train
case066.png,BrEaST,malignant,845,train
case067.png,BrEaST,malignant,846,train
case068.png,BrEaST,benign,847,train
case069.png,BrEaST,benign,848,test
case070.png,BrEaST,benign,849,train
case071.png,BrEaST,benign,850,val
case072.png,BrEaST,malignant,851,train
case073.png,BrEaST,benign,852,train
case074.png,BrEaST,benign,853,train
case075.png,BrEaST,malignant,854,train
case076.png,BrEaST,malignant,855,test
case077.png,BrEaST,benign,856,val
case078.png,BrEaST,benign,857,train
case079.png,BrEaST,malignant,858,train
case080.png,BrEaST,benign,859,train
case081.png,BrEaST,benign,860,train
case082.png,BrEaST,malignant,861,train
case083.png,BrEaST,benign,862,train
case084.png,BrEaST,benign,863,val
case085.png,BrEaST,benign,864,train
case086.png,BrEaST,malignant,865,test
case087.png,BrEaST,malignant,866,train
case088.png,BrEaST,malignant,867,train
case089.png,BrEaST,malignant,868,train
case090.png,BrEaST,malignant,869,train
case091.png,BrEaST,benign,870,test
case092.png,BrEaST,benign,871,train
case093.png,BrEaST,benign,872,train
case094.png,BrEaST,malignant,873,val
case095.png,BrEaST,benign,874,train
case096.png,BrEaST,benign,875,train
case097.png,BrEaST,benign,876,train
case098.png,BrEaST,benign,877,train
case099.png,BrEaST,malignant,878,train
case100.png,BrEaST,benign,879,train
case101.png,BrEaST,malignant,880,test
case102.png,BrEaST,malignant,881,test
case103.png,BrEaST,malignant,882,val
case104.png,BrEaST,benign,883,val
case105.png,BrEaST,malignant,884,train
case106.png,BrEaST,malignant,885,train
case107.png,BrEaST,malignant,886,train
case108.png,BrEaST,benign,887,train
case109.png,BrEaST,malignant,888,train
case110.png,BrEaST,benign,889,train
case111.png,BrEaST,malignant,890,train
case112.png,BrEaST,malignant,891,test
case113.png,BrEaST,malignant,892,train
case114.png,BrEaST,malignant,893,train
case115.png,BrEaST,benign,894,train
case116.png,BrEaST,benign,895,train
case117.png,BrEaST,malignant,896,train
case118.png,BrEaST,benign,897,train
case119.png,BrEaST,benign,898,val
case120.png,BrEaST,benign,899,train
case121.png,BrEaST,malignant,900,test
case122.png,BrEaST,benign,901,train
case123.png,BrEaST,benign,902,val
case124.png,BrEaST,benign,903,train
case125.png,BrEaST,malignant,904,train
case126.png,BrEaST,benign,905,train
case127.png,BrEaST,malignant,906,train
case128.png,BrEaST,malignant,907,train
case129.png,BrEaST,benign,908,test
case130.png,BrEaST,malignant,909,val
case131.png,BrEaST,benign,910,train
case132.png,BrEaST,malignant,911,train
case133.png,BrEaST,malignant,912,train
case134.png,BrEaST,benign,913,train
case135.png,BrEaST,benign,914,val
case136.png,BrEaST,benign,915,train
case137.png,BrEaST,benign,916,train
case138.png,BrEaST,benign,917,train
case139.png,BrEaST,benign,918,test
case140.png,BrEaST,benign,919,train
case141.png,BrEaST,benign,920,train
case142.png,BrEaST,benign,921,train
case143.png,BrEaST,benign,922,train
case144.png,BrEaST,malignant,923,train
case145.png,BrEaST,malignant,924,test
case146.png,BrEaST,benign,925,train
case147.png,BrEaST,malignant,926,val
case148.png,BrEaST,malignant,927,train
case149.png,BrEaST,malignant,928,train
case150.png,BrEaST,malignant,929,train
case151.png,BrEaST,benign,930,train
case152.png,BrEaST,benign,931,test
case153.png,BrEaST,malignant,932,val
case154.png,BrEaST,malignant,933,train
case155.png,BrEaST,benign,934,test
case156.png,BrEaST,benign,935,train
case157.png,BrEaST,benign,936,train
case158.png,BrEaST,malignant,937,val
case159.png,BrEaST,benign,938,train
case160.png,BrEaST,malignant,939,train
case161.png,BrEaST,benign,940,train
case162.png,BrEaST,benign,941,train
case163.png,BrEaST,benign,942,test
case164.png,BrEaST,malignant,943,train
case165.png,BrEaST,benign,944,train
case166.png,BrEaST,benign,945,val
case167.png,BrEaST,benign,946,train
case168.png,BrEaST,benign,947,train
case169.png,BrEaST,benign,948,train
case170.png,BrEaST,malignant,949,train
case171.png,BrEaST,benign,950,train
case172.png,BrEaST,malignant,951,test
case173.png,BrEaST,malignant,952,val
case174.png,BrEaST,malignant,953,test
case175.png,BrEaST,benign,954,train
case176.png,BrEaST,malignant,955,train
case177.png,BrEaST,benign,956,train
case178.png,BrEaST,benign,957,train
case179.png,BrEaST,benign,958,train
case180.png,BrEaST,benign,959,train
case181.png,BrEaST,benign,960,train
case182.png,BrEaST,benign,961,test
case183.png,BrEaST,benign,962,val
case184.png,BrEaST,benign,963,train
case185.png,BrEaST,benign,964,train
case186.png,BrEaST,benign,965,train
case187.png,BrEaST,benign,966,train
case188.png,BrEaST,benign,967,train
case189.png,BrEaST,benign,968,test
case190.png,BrEaST,malignant,969,train
case191.png,BrEaST,benign,970,train
case192.png,BrEaST,benign,971,val
case193.png,BrEaST,benign,972,val
case194.png,BrEaST,benign,973,train
case195.png,BrEaST,malignant,974,train
case196.png,BrEaST,benign,975,test
case197.png,BrEaST,benign,976,val
case198.png,BrEaST,malignant,977,train
case199.png,BrEaST,malignant,978,test
case200.png,BrEaST,benign,979,train
case201.png,BrEaST,benign,980,train
case202.png,BrEaST,benign,981,train
case203.png,BrEaST,benign,982,train
case204.png,BrEaST,benign,983,train
case205.png,BrEaST,malignant,984,test
case206.png,BrEaST,malignant,985,train
case207.png,BrEaST,benign,986,val
case208.png,BrEaST,benign,987,train
case209.png,BrEaST,normal,988,train
case210.png,BrEaST,benign,989,train
case211.png,BrEaST,malignant,990,train
case212.png,BrEaST,malignant,991,val
case213.png,BrEaST,normal,992,train
case214.png,BrEaST,benign,993,train
case215.png,BrEaST,benign,994,train
case216.png,BrEaST,benign,995,test
case217.png,BrEaST,benign,996,train
case218.png,BrEaST,malignant,997,val
case219.png,BrEaST,malignant,998,train
case220.png,BrEaST,benign,999,train
case221.png,BrEaST,malignant,1000,train
case222.png,BrEaST,benign,1001,train
case223.png,BrEaST,benign,1002,test
case224.png,BrEaST,benign,1003,train
case225.png,BrEaST,malignant,1004,train
case226.png,BrEaST,malignant,1005,train
case227.png,BrEaST,malignant,1006,val
case228.png,BrEaST,malignant,1007,train
case229.png,BrEaST,benign,1008,test
case230.png,BrEaST,malignant,1009,train
case231.png,BrEaST,benign,1010,train
case232.png,BrEaST,benign,1011,train
case233.png,BrEaST,malignant,1012,val
case234.png,BrEaST,malignant,1013,train
case235.png,BrEaST,benign,1014,train
case236.png,BrEaST,benign,1015,val
case237.png,BrEaST,benign,1016,train
case238.png,BrEaST,malignant,1017,train
case239.png,BrEaST,malignant,1018,train
case240.png,BrEaST,benign,1019,train
case241.png,BrEaST,benign,1020,test
case242.png,BrEaST,benign,1021,val
case243.png,BrEaST,malignant,1022,train
case244.png,BrEaST,malignant,1023,train
case245.png,BrEaST,malignant,1024,train
case246.png,BrEaST,benign,1025,train
case247.png,BrEaST,malignant,1026,train
case248.png,BrEaST,malignant,1027,test
case249.png,BrEaST,malignant,1028,train
case250.png,BrEaST,malignant,1029,train
case251.png,BrEaST,malignant,1030,train
case252.png,BrEaST,benign,1031,train
case253.png,BrEaST,benign,1032,train
case254.png,BrEaST,malignant,1033,val
case255.png,BrEaST,malignant,1034,train
case256.png,BrEaST,benign,1035,train
'''

## 1. Load the three datasets

In [ ]:
# BUSI (Kaggle): <class>/<name>.png, lesion masks are <name>_mask.png, <name>_mask_1.png, ...
busi_root = glob.glob(f"{INPUT}/**/Dataset_BUSI_with_GT", recursive=True)[0]
rows = []
for label in CLASSES:
    for path in sorted(glob.glob(f"{busi_root}/{label}/*.png")):
        if "_mask" not in os.path.basename(path):
            masks = sorted(glob.glob(glob.escape(path[:-4]) + "_mask*.png"))
            rows.append({"path": path, "source": "BUSI", "label": label, "masks": masks})

# BrEaST-Lesions-USG (TCIA, CC BY 4.0): one scan per patient, labels confirmed by biopsy or follow-up
TCIA = "https://www.cancerimagingarchive.net/wp-content/uploads/"
zip_path = download(TCIA + "BrEaST-Lesions_USG-images_and_masks-Dec-15-2023.zip", f"{TMP}/breast_usg.zip")
xlsx_path = download(TCIA + "BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx", f"{TMP}/breast_usg.xlsx")
if not glob.glob(f"{TMP}/breast_usg/**/case001.png", recursive=True):
    zipfile.ZipFile(zip_path).extractall(f"{TMP}/breast_usg")
img_dir = os.path.dirname(glob.glob(f"{TMP}/breast_usg/**/case001.png", recursive=True)[0])
clinical = pd.read_excel(xlsx_path)
for r in clinical.itertuples():
    masks = [f"{img_dir}/{m}" for m in r.Mask_tumor_filename.split("&")] if isinstance(r.Mask_tumor_filename, str) else []
    rows.append({"path": f"{img_dir}/{r.Image_filename}", "source": "BrEaST", "label": r.Classification, "masks": masks})

# BUS-BRA (Kaggle mirror of Gómez-Flores et al., Medical Physics 2024): National Cancer Institute, Rio de Janeiro,
# 4 scanners, 1,875 biopsy-proven scans from 1,064 patients (benign / malignant only, no normal class).
bra_dir = os.path.dirname(glob.glob(f"{INPUT}/**/bus_data.csv", recursive=True)[0])
for r in pd.read_csv(f"{bra_dir}/bus_data.csv").itertuples():
    mask = f"{bra_dir}/Masks/mask_{r.ID[4:]}.png"
    rows.append({"path": f"{bra_dir}/Images/{r.ID}.png", "source": "BUS-BRA", "label": r.Pathology,
                 "masks": [mask] if os.path.exists(mask) else [], "case": f"bra{r.Case}", "device": r.Device})

df = pd.DataFrame(rows)
assert set(df["label"]) <= set(CLASSES), set(df["label"])
if SMOKE:
    df = df.groupby(["source", "label"], group_keys=False).head(12).reset_index(drop=True)
print(pd.crosstab(df["source"], df["label"], margins=True))

fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for ax in axes.flat:
    ax.axis("off")
for ax, r in zip(axes.T.flat, df.groupby(["source", "label"]).head(2).itertuples()):
    ax.imshow(Image.open(r.path).convert("L"), cmap="gray")
    ax.set_title(f"{r.source} · {r.label}", fontsize=10)
plt.tight_layout(); plt.show()

## 2. Remove near-duplicates, then split by duplicate group

Each image gets a 256-bit difference hash. Images whose hashes differ in ≤ 12 bits (≈ 5%) are treated as copies of the same
scan and kept in the same split. Copies with *conflicting* labels are dropped, because we can't tell which label is correct.
BUSI and BrEaST keep v7's exact split. BUS-BRA is split by patient (both breasts of a patient stay together).

In [ ]:
def dhash(path, size=16):
    g = np.asarray(Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()

h = np.stack([dhash(p) for p in df["path"]]).astype(np.float32) * 2 - 1
hamming = (h.shape[1] - h @ h.T) / 2
parent = list(range(len(df)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i, j in zip(*np.where(np.triu(hamming <= 12, k=1))):
    parent[find(i)] = find(j)
df["group"] = [find(i) for i in range(len(df))]

dups = df[df.duplicated("group", keep=False)]
n_labels = dups.groupby("group")["label"].nunique()
conflicting = n_labels[n_labels > 1].index
print(f"{len(dups)} images fall into {dups['group'].nunique()} near-duplicate groups; "
      f"{len(conflicting)} groups have conflicting labels and are dropped "
      f"({df['group'].isin(conflicting).sum()} images)")

examples = dups.groupby("group").head(2).head(8)
if len(examples):
    fig, axes = plt.subplots(1, len(examples), figsize=(2.4 * len(examples), 2.8))
    for ax, r in zip(np.atleast_1d(axes), examples.itertuples()):
        ax.imshow(Image.open(r.path).convert("L"), cmap="gray"); ax.axis("off")
        ax.set_title(f"group {r.group}\n{r.label}", fontsize=8)
    plt.suptitle("Examples of detected near-duplicates (pairs)"); plt.tight_layout(); plt.show()

dedup_stats = {"images_in_duplicate_groups": int(len(dups)), "duplicate_groups": int(dups["group"].nunique()),
               "dropped_conflicting_images": int(df["group"].isin(conflicting).sum())}
df = df[~df["group"].isin(conflicting)].reset_index(drop=True)
df["strata"] = df["source"] + "_" + df["label"]

# ---- BUSI + BrEaST keep the exact train/val/test assignment of notebook v7, so v7's test set is unchanged.
# BUS-BRA is split by patient. All BUS-BRA test patients are unseen by the model, and come from a third hospital.
V7 = pd.read_csv(io.StringIO(V7_SPLIT_CSV))
v7_split = dict(zip(V7["source"] + "/" + V7["file"], V7["split"]))
old = df["source"].isin(["BUSI", "BrEaST"]).values
df["split"] = (df["source"] + "/" + df["path"].map(os.path.basename)).map(v7_split)
df = df[~(old & df["split"].isna().values)].reset_index(drop=True)   # v7 dropped these (conflicting near-duplicates)

bra = (df["source"] == "BUS-BRA").values
shared = df.groupby("group")["source"].transform(lambda s: bool((s == "BUS-BRA").any() and (s != "BUS-BRA").any()))
print(f"{int((shared.values & bra).sum())} BUS-BRA images are near-duplicates of BUSI/BrEaST images and are dropped")
df = df[~(shared.values & bra)].reset_index(drop=True)
bra = (df["source"] == "BUS-BRA").values

# One patient (both breasts) and any near-duplicate scans always stay together, in the split and in cross-validation
uf = {}
def ufind(a):
    uf.setdefault(a, a)
    while uf[a] != a:
        uf[a] = uf[uf[a]]
        a = uf[a]
    return a
for case, grp in zip(df.loc[bra, "case"], df.loc[bra, "group"]):
    uf[ufind(case)] = ufind(f"g{grp}")
df.loc[bra, "group"] = pd.factorize(pd.Series([ufind(c) for c in df.loc[bra, "case"]]))[0] + 10**6

bra_idx = np.where(bra)[0]
if SMOKE:
    df.loc[bra, "split"] = np.array(["test", "val", "train"])[np.arange(len(bra_idx)) % 3]
else:
    fold = np.zeros(len(bra_idx), int)
    for k, (_, part) in enumerate(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
                                  .split(bra_idx, df["strata"].values[bra_idx], df["group"].values[bra_idx])):
        fold[part] = k
    df.loc[bra, "split"] = np.select([fold < 3, fold == 3], ["test", "val"], "train")   # 30% / 10% / 60% of patients
idx = {s: np.where(df["split"] == s)[0] for s in ["train", "val", "test"]}
print(pd.crosstab([df["source"], df["label"]], df["split"], margins=True))

## 3. Preprocessing, augmentation and the two training recipes

- **baseline**: mild augmentation; classes weighted by inverse frequency.
- **source-balanced + scanner augmentation**: BUSI makes up ~75% of the images, so a model can score well by learning
  what BUSI's scanners look like. This recipe gives each hospital equal total weight and adds augmentation that mimics
  differences between machines (zoom, blur, sharpness, gain and contrast).

In [ ]:
labels = df["label"].map(CLASSES.index).values
sources = df["source"].values
devices = df["device"].fillna("").values
gray224 = np.stack([to_gray224(Image.open(p)) for p in df["path"]])   # exactly what the backend feeds the model
train_imgs = [Image.fromarray(np.asarray(pad_square(ImageOps.exif_transpose(Image.open(p)).convert("L"))
                                         .resize((288, 288), Image.BILINEAR))) for p in df["path"]]

AUGMENT = {
    "standard": transforms.Compose([
        transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
        transforms.RandomHorizontalFlip(),   # no vertical flip: the transducer is always at the top of the image
        transforms.RandomApply([transforms.RandomRotation(10)], p=0.5),
        transforms.ColorJitter(brightness=0.25, contrast=0.25),
        transforms.PILToTensor(),
    ]),
    "scanner": transforms.Compose([
        transforms.RandomResizedCrop(IMG, scale=(0.55, 1.0), ratio=(0.8, 1.25)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.RandomRotation(12)], p=0.5),
        transforms.ColorJitter(brightness=0.4, contrast=0.4),
        transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
        transforms.RandomAdjustSharpness(2.0, p=0.3),
        transforms.PILToTensor(),
    ]),
}
RECIPES = {
    "baseline": {"augment": "standard", "balance_sources": False},
    "source-balanced + scanner augmentation": {"augment": "scanner", "balance_sources": True},
}

def sample_weights(ids, balance_sources):
    # inverse class frequency, optionally times inverse source frequency so both hospitals count equally
    y, src = labels[ids], sources[ids]
    w = (len(ids) / (len(CLASSES) * np.maximum(np.bincount(y, minlength=len(CLASSES)), 1)))[y]
    if balance_sources:
        names, counts = np.unique(src, return_counts=True)
        per_source = dict(zip(names, len(ids) / (len(names) * counts)))
        w = w * np.array([per_source[s] for s in src])
    out = np.zeros(len(df), np.float32)
    out[ids] = w / w.mean()
    return out

class UltrasoundDS(Dataset):
    def __init__(self, ids, augment=None):
        self.ids, self.augment = np.asarray(ids), augment
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]
        g = AUGMENT[self.augment](train_imgs[j])[0].numpy() if self.augment else gray224[j]
        return torch.from_numpy(normalize(g)), int(labels[j]), int(j)

def loader(ids, augment=None):
    return DataLoader(UltrasoundDS(ids, augment), batch_size=BATCH, shuffle=augment is not None,
                      drop_last=augment is not None, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])   # -> B x 2048 x 7 x 7 feature maps
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(2048, len(CLASSES))
    def forward(self, x):
        emb = self.backbone(x).mean((2, 3))   # global average pooling
        return self.fc(self.drop(emb)), emb

def predict_logits(net, ids, amp=True):
    net.eval()
    out = []
    with torch.no_grad():
        for x, *_ in loader(ids):
            with torch.autocast(DEVICE.type, enabled=amp and DEVICE.type == "cuda"):
                out.append(net(x.to(DEVICE))[0].float().cpu())
    return torch.cat(out).numpy()

def summarize(y, p, pred=None):
    pred = p.argmax(1) if pred is None else pred
    return {"accuracy": round(float(accuracy_score(y, pred)), 4),
            "macro_f1": round(float(f1_score(y, pred, average="macro")), 4),
            "malignant_sensitivity": round(float(recall_score(y == MAL, pred == MAL)), 4),
            "malignant_specificity": round(float(recall_score(y != MAL, pred != MAL)), 4),
            "malignant_auc": round(float(roc_auc_score(y == MAL, p[:, MAL])), 4)}

def summarize_by_source(ids, p, pred=None):
    out = {}
    for src in SOURCES:
        m = sources[ids] == src
        try:
            out[src] = {"images": int(m.sum()), **summarize(labels[ids][m], p[m], None if pred is None else pred[m])}
        except ValueError as e:   # e.g. a class missing from a tiny smoke-test split
            print(src, e)
    return out

fig, axes = plt.subplots(2, 6, figsize=(15, 5.6))
for row, name in zip(axes, AUGMENT):
    for ax in row:
        ax.imshow(AUGMENT[name](train_imgs[idx["train"][0]])[0], cmap="gray"); ax.axis("off")
    row[0].set_title(f"{name} augmentation", loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Baseline: frozen ImageNet ResNet50 + logistic regression

A *linear probe* uses ResNet50 exactly as trained on everyday photos, with no fine-tuning. It shows how much the
fine-tuning in the next sections actually adds.

In [ ]:
@torch.no_grad()
def embed(net, ids):
    net.eval()
    return torch.cat([net(x.to(DEVICE))[1].float().cpu() for x, *_ in loader(ids)]).numpy()

imagenet_net = Net().to(DEVICE)
probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=0.05, class_weight="balanced"))
probe.fit(embed(imagenet_net, idx["train"]), labels[idx["train"]])
p_probe_test = probe.predict_proba(embed(imagenet_net, idx["test"]))
baseline = summarize(labels[idx["test"]], p_probe_test)
del imagenet_net
print(json.dumps(baseline, indent=2))

## 5. Choose the training recipe by 5-fold cross-validation (development set only)

Phase 1 trains only the new classifier head (backbone frozen). Phase 2 unfreezes the whole network, with a 5× lower
learning rate for the pretrained layers and cosine decay. Folds use the last epoch (no checkpoint picking), so the
estimate is not optimistically biased. The test set is not touched.

**Selection rule (fixed before running):** the recipe with the higher average of the out-of-fold malignant ROC-AUC on
BUSI and on BrEaST, so doing well on one hospital cannot hide doing badly on the other.

In [ ]:
def train_model(train_ids, val_ids, recipe, select_best=True, verbose=False):
    seed_everything()
    net = Net().to(DEVICE)
    weights = torch.tensor(sample_weights(train_ids, recipe["balance_sources"]), device=DEVICE)
    criterion = nn.CrossEntropyLoss(reduction="none", label_smoothing=0.05)
    train_dl = loader(train_ids, recipe["augment"])
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

    for p in net.backbone.parameters():
        p.requires_grad = False
    opt, sched = torch.optim.AdamW(net.fc.parameters(), lr=1e-3, weight_decay=1e-4), None
    best, history = (-1.0, None, 0), []
    for epoch in range(EPOCHS_HEAD + EPOCHS_FT):
        if epoch == EPOCHS_HEAD:   # phase 2: fine-tune everything
            for p in net.backbone.parameters():
                p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.backbone.parameters(), "lr": 1e-4},
                                     {"params": net.fc.parameters(), "lr": 5e-4}], weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FT * len(train_dl))
        net.train()
        loss_sum, seen = 0.0, 0
        for x, y, j in train_dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = (criterion(net(x)[0], y) * weights[j.to(DEVICE)]).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if sched:
                sched.step()
            loss_sum, seen = loss_sum + loss.item() * len(y), seen + len(y)
        val_f1 = f1_score(labels[val_ids], predict_logits(net, val_ids).argmax(1), average="macro")
        history.append({"epoch": epoch + 1, "train_loss": loss_sum / max(seen, 1), "val_macro_f1": val_f1})
        if val_f1 > best[0]:
            best = (val_f1, copy.deepcopy(net.state_dict()), epoch + 1)
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {history[-1]['train_loss']:.3f}  val macro-F1 {val_f1:.3f}")
    if select_best:
        net.load_state_dict(best[1])
    return net, pd.DataFrame(history), best[2]

dev = np.concatenate([idx["train"], idx["val"]])
cv_splits = list(StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
                 .split(dev, df["strata"].values[dev], df["group"].values[dev]))
oof_logits, cv_rows = {}, []
for name, recipe in RECIPES.items():
    oof_logits[name] = np.zeros((len(df), len(CLASSES)), np.float32)
    for k, (tr, va) in enumerate(cv_splits):
        fold_net, _, _ = train_model(dev[tr], dev[va], recipe, select_best=False)
        logits = predict_logits(fold_net, dev[va], amp=False)
        oof_logits[name][dev[va]] = logits
        cv_rows.append({"recipe": name, "fold": k + 1, **summarize(labels[dev[va]], softmax(logits))})
        print(cv_rows[-1])
        del fold_net
        torch.cuda.empty_cache()
cv_table = pd.DataFrame(cv_rows)

recipe_rows = []
for name in RECIPES:
    p_oof = softmax(oof_logits[name][dev])
    by_source = summarize_by_source(dev, p_oof)
    folds_of = cv_table[cv_table.recipe == name]
    recipe_rows.append({
        "recipe": name,
        "selection_score": round(float(np.mean([by_source[s]["malignant_auc"] for s in by_source])), 4),
        **{f"cv_{m}": f"{folds_of[m].mean():.3f} ± {folds_of[m].std():.3f}"
           for m in ["accuracy", "macro_f1", "malignant_sensitivity", "malignant_auc"]},
        **{f"{s}_{m}": by_source[s][m] for s in by_source for m in ["accuracy", "malignant_auc"]},
    })
recipe_table = pd.DataFrame(recipe_rows)
BEST_RECIPE = recipe_table.loc[recipe_table["selection_score"].idxmax(), "recipe"]
display(recipe_table)
print("Selected recipe:", BEST_RECIPE)
cv_summary = recipe_table.set_index("recipe").loc[BEST_RECIPE].to_dict()

## 6. Final model (selected recipe, train split, best epoch picked on the validation split)

In [ ]:
net, history, best_epoch = train_model(idx["train"], idx["val"], RECIPES[BEST_RECIPE], verbose=True)
p_torch_test = softmax(predict_logits(net, idx["test"], amp=False))   # full-precision reference for the parity check

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(history["epoch"], history["train_loss"], color="#C2185B", label="train loss")
ax2 = ax1.twinx()
ax2.plot(history["epoch"], history["val_macro_f1"], color="#6C2D7E", label="val macro-F1")
ax1.axvline(best_epoch, ls="--", color="grey"); ax1.axvline(EPOCHS_HEAD + 0.5, ls=":", color="lightgrey")
ax1.set(xlabel="epoch", ylabel="train loss"); ax2.set_ylabel("val macro-F1")
fig.legend(loc="upper center", ncol=2); plt.title(f"Training (best epoch {best_epoch})", pad=24)
plt.tight_layout(); plt.savefig(f"{OUT}/breast_training_curve.png", dpi=150); plt.show()

## 7. Export to ONNX and verify parity

The exported graph returns three outputs: **logits** (the prediction), the **embedding** (used by the ultrasound gate)
and **class activation maps** (the heatmap). With a global-average-pool + linear head, CAM is exact: it is the
classifier's weights applied to every spatial location of the last feature map.

The fp32 ResNet50 is 94 MB, close to GitHub's 100 MB file limit. Conv/linear weights are therefore **stored** in fp16 and
cast back to fp32 when the model loads. All computation stays in fp32.

In [ ]:
import onnx, onnxruntime as ort
from onnx import helper, numpy_helper, TensorProto

class ExportNet(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, x):
        fmap = self.net.backbone(x)
        emb = fmap.mean((2, 3))
        cam = F.conv2d(fmap, self.net.fc.weight[:, :, None, None])   # class activation maps, B x 3 x 7 x 7
        return self.net.fc(emb), emb, cam

net = net.float().cpu().eval()
fp32_path = f"{TMP}/breast_resnet50_fp32.onnx"
export_args = dict(input_names=["image"], output_names=["logits", "embedding", "cam"], opset_version=17,
                   dynamic_axes={n: {0: "batch"} for n in ["image", "logits", "embedding", "cam"]})
dummy = torch.from_numpy(normalize(gray224[0]))[None]
try:
    torch.onnx.export(ExportNet(net).eval(), dummy, fp32_path, dynamo=False, **export_args)
except TypeError:   # torch < 2.5 has no `dynamo` argument
    torch.onnx.export(ExportNet(net).eval(), dummy, fp32_path, **export_args)

model = onnx.load(fp32_path)
graph = model.graph
initializers, casts = [], []
for init in graph.initializer:
    w = numpy_helper.to_array(init)
    if w.dtype == np.float32 and w.ndim >= 2:   # conv / linear weights; BatchNorm statistics stay fp32
        initializers.append(numpy_helper.from_array(w.astype(np.float16), init.name + "_fp16"))
        casts.append(helper.make_node("Cast", [init.name + "_fp16"], [init.name], to=TensorProto.FLOAT))
    else:
        initializers.append(numpy_helper.from_array(w, init.name))
nodes = casts + [copy.deepcopy(n) for n in graph.node]
graph.ClearField("initializer"); graph.initializer.extend(initializers)
graph.ClearField("node"); graph.node.extend(nodes)
onnx.checker.check_model(model)
ONNX_PATH = f"{OUT}/breast_resnet50.onnx"
onnx.save(model, ONNX_PATH)
print(f"fp32 {os.path.getsize(fp32_path) / 1e6:.1f} MB -> deployed {os.path.getsize(ONNX_PATH) / 1e6:.1f} MB")

def run_session(sess, grays, batch=32):
    outs = [sess.run(None, {"image": np.stack([normalize(g) for g in grays[i:i + batch]])})
            for i in range(0, len(grays), batch)]
    return [np.concatenate(o) for o in zip(*outs)]

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
def run_onnx(grays):
    return run_session(session, grays)

onnx_out = {s: run_onnx(gray224[idx[s]]) for s in ["train", "val", "test"]}   # (logits, embedding, cam) per split
# Two separate checks: the export itself must be exact; fp16 weight storage only adds rounding noise.
# (All metrics below are computed with the deployed model, so that noise is already reflected in them.)
fp32_session = ort.InferenceSession(fp32_path, providers=["CPUExecutionProvider"])
p_onnx_test = softmax(onnx_out["test"][0])
parity_fp32 = float(np.abs(softmax(run_session(fp32_session, gray224[idx["test"]])[0]) - p_torch_test).max())
parity = float(np.abs(p_onnx_test - p_torch_test).max())
flipped = int((p_onnx_test.argmax(1) != p_torch_test.argmax(1)).sum())
print(f"max |P_onnx - P_torch| on the test set: fp32 export {parity_fp32:.5f}, deployed (fp16 weights) {parity:.5f}; "
      f"{flipped} of {len(p_torch_test)} test predictions change class")
assert parity_fp32 < 1e-3, "ONNX export disagrees with the trained network"
assert parity < 0.05, "fp16 weight storage changes predictions too much"

## 8. Calibration and the screening threshold (out-of-fold predictions)

- **Temperature scaling** makes the displayed confidence honest: when the app says "85% confident", it should be right
  about 85% of the time.
- **Screening rule**: a scan is flagged *suspicious* when P(malignant) ≥ threshold, even if another class is more likely.
  The threshold is the highest value that still catches ≥ 90% of malignant scans.

Both are fitted on the selected recipe's out-of-fold predictions for the whole development set (~870 scans, ~260
malignant). A threshold fitted on the small validation split alone (~30 malignant scans) is too noisy.

In [ ]:
def fit_temperature(logits, y):
    log_t = torch.zeros(1, requires_grad=True)
    z, target = torch.tensor(logits), torch.tensor(y)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(z / log_t.exp(), target)
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_t.exp())

def ece(p, y, bins=10):
    # expected calibration error: average gap between confidence and accuracy, weighted by bin size
    conf, correct = p.max(1), p.argmax(1) == y
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)

def decide(p, t_mal):
    pred = p.argmax(1)
    pred[p[:, MAL] >= t_mal] = MAL
    return pred

z_oof, y_dev = oof_logits[BEST_RECIPE][dev], labels[dev]
TEMPERATURE = fit_temperature(z_oof, y_dev)
p_oof = softmax(z_oof / TEMPERATURE)
ok = [t for t in np.round(np.arange(0.50, 0.04, -0.01), 2)
      if recall_score(y_dev == MAL, decide(p_oof, t) == MAL) >= 0.90]
MAL_THRESHOLD = float(ok[0]) if ok else 0.10
y_val, y_test = labels[idx["val"]], labels[idx["test"]]
calibration = {"temperature": round(TEMPERATURE, 4), "fitted_on": f"{len(dev)} out-of-fold predictions",
               "oof_ece_before": round(ece(softmax(z_oof), y_dev), 4), "oof_ece_after": round(ece(p_oof, y_dev), 4),
               "val_ece_final_model": round(ece(softmax(onnx_out["val"][0] / TEMPERATURE), y_val), 4)}
print(calibration, "| malignant threshold:", MAL_THRESHOLD)

## 9. Held-out test set (computed with the exported ONNX model)

In [ ]:
p_test = softmax(onnx_out["test"][0] / TEMPERATURE)
pred_test = decide(p_test, MAL_THRESHOLD)
test_metrics = {"test_size": int(len(y_test)), **summarize(y_test, p_test, pred_test),
                "macro_auc_ovr": round(float(roc_auc_score(y_test, p_test, multi_class="ovr")), 4),
                "ece": round(ece(p_test, y_test), 4)}
calibration["test_ece_before"] = round(ece(softmax(onnx_out["test"][0]), y_test), 4)
calibration["test_ece_after"] = test_metrics["ece"]
argmax_metrics = summarize(y_test, p_test)
print(json.dumps(test_metrics, indent=2))
print(classification_report(y_test, pred_test, target_names=CLASSES))

per_source = summarize_by_source(idx["test"], p_test, pred_test)
print(pd.DataFrame(per_source).T)

# Same images as v7's test set (BUSI + BrEaST), so this row is directly comparable with v7's published numbers
old_t = np.isin(sources[idx["test"]], ["BUSI", "BrEaST"])
test_v7_subset = {"test_size": int(old_t.sum()), **summarize(y_test[old_t], p_test[old_t], pred_test[old_t])}
test_busbra = {"test_size": int((~old_t).sum()), **summarize(y_test[~old_t], p_test[~old_t], pred_test[~old_t])}
per_device = {}
for d in sorted(set(devices[idx["test"]][~old_t])):
    m = devices[idx["test"]] == d
    try:
        per_device[d] = {"images": int(m.sum()), **summarize(y_test[m], p_test[m], pred_test[m])}
    except ValueError as e:
        print(d, e)
print("v7-comparable test subset:", test_v7_subset)
print("BUS-BRA held-out patients:", test_busbra)
display(pd.DataFrame(per_device).T)

comparison = pd.DataFrame([
    {"model": "Linear probe (frozen ImageNet ResNet50)", **baseline},
    {"model": "Fine-tuned ResNet50, argmax", **argmax_metrics},
    {"model": f"Fine-tuned ResNet50, screening rule (P(mal) >= {MAL_THRESHOLD})", **summarize(y_test, p_test, pred_test)},
])
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sns.heatmap(confusion_matrix(y_test, pred_test, labels=range(len(CLASSES))), annot=True, fmt="d", cmap="RdPu",
            ax=axes[0], xticklabels=CLASSES, yticklabels=CLASSES)
axes[0].set(title="Confusion matrix (test set, screening rule)", xlabel="Predicted", ylabel="Actual")
for src, colour in [("BUSI", "#C2185B"), ("BrEaST", "#6C2D7E"), ("BUS-BRA", "#00897B")]:
    m = sources[idx["test"]] == src
    fpr, tpr, _ = roc_curve(y_test[m] == MAL, p_test[m, MAL])
    axes[1].plot(fpr, tpr, color=colour, label=f"{src} (AUC {per_source[src]['malignant_auc']:.2f})")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="Malignant vs rest, by hospital")
axes[1].legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT}/breast_evaluation.png", dpi=150); plt.show()

## 10. Explainability: do the heatmaps land on the lesion?

For every test scan with a lesion, the activation map of its true class is upsampled to the image. Two scores:
- **Pointing game**: is the hottest pixel inside the radiologist's lesion mask?
- **IoU**: overlap between the hot region (≥ 50% of the maximum) and the mask.

In [ ]:
def upsample_cam(cam7):
    c = np.maximum(cam7, 0)
    c = c / c.max() if c.max() > 0 else c
    return np.asarray(Image.fromarray((c * 255).astype(np.uint8)).resize((IMG, IMG), Image.BILINEAR)) / 255.0

def mask224(paths):
    m = np.zeros((IMG, IMG), bool)
    for p in paths:
        m |= np.asarray(pad_square(Image.open(p).convert("L")).resize((IMG, IMG), Image.NEAREST)) > 127
    return m

cam_test = onnx_out["test"][2]
hits, ious, chance, shown = [], [], [], []
for k, j in enumerate(idx["test"]):
    if labels[j] == CLASSES.index("normal") or not df.at[j, "masks"]:
        continue
    mask = mask224(df.at[j, "masks"])
    if not mask.any():
        continue
    heat = upsample_cam(cam_test[k, labels[j]])
    hits.append(bool(mask[np.unravel_index(heat.argmax(), heat.shape)]))
    hot = heat >= 0.5
    ious.append((hot & mask).sum() / (hot | mask).sum())
    chance.append(mask.sum() / (gray224[j] > 10).sum())   # hit rate of a random point on the scan
    if len(shown) < 8 and (len(shown) < 4 or labels[j] == MAL):
        shown.append((j, heat, mask))
cam_eval = {"lesion_images": len(hits), "pointing_game": round(float(np.mean(hits)), 4),
            "mean_iou": round(float(np.mean(ious)), 4), "pointing_game_chance": round(float(np.mean(chance)), 4)}
print(cam_eval)

fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax in axes.flat:
    ax.axis("off")
for ax, (j, heat, mask) in zip(axes.flat, shown):
    ax.imshow(gray224[j], cmap="gray"); ax.imshow(heat, cmap="jet", alpha=0.35)
    ax.contour(mask, levels=[0.5], colors="white", linewidths=1.2)
    ax.set_title(f"{df.at[j, 'label']} ({df.at[j, 'source']})", fontsize=10)
plt.suptitle("Class activation maps (colour) vs radiologist lesion masks (white outline)")
plt.tight_layout(); plt.savefig(f"{OUT}/breast_cam_examples.png", dpi=130); plt.show()

## 11. Rejecting uploads that aren't breast ultrasounds

People will upload the wrong image: a selfie, a photo of a report, an X-ray. The classifier would still output
normal/benign/malignant for any of these, so the backend runs two checks first:
1. **Colour check**: B-mode ultrasound is grey. Images where more than 10% of pixels are clearly coloured are rejected.
2. **Ultrasound gate**: a logistic regression on the network's embedding, trained to separate the training ultrasounds
   from *grayscale* photos (4 categories) and chest X-rays. Its threshold lets 99% of validation scans through.

To check that the gate generalises, it is tested on held-out ultrasounds and on images it never saw: other photo
categories (in colour and in grayscale), held-out chest X-rays, and brain MRIs, an image type absent from its training.
(An unsupervised feature-distance check was tried first and rejected almost nothing: photos and X-rays landed at the
same distances as real scans.)

In [ ]:
def find_images(pattern):
    paths = glob.glob(f"{INPUT}/**/{pattern}", recursive=True)
    return sorted(p for p in paths if "__MACOSX" not in p and not os.path.basename(p).startswith("._")
                  and p.lower().endswith((".jpg", ".jpeg", ".png")))

def load_images(paths):
    images = []
    for p in paths:
        try:
            im = Image.open(p)
            im.load()
            images.append(im)
        except Exception:
            pass
    return images

rng = np.random.default_rng(SEED)
def sample(paths, n):
    return list(rng.choice(paths, min(n, len(paths)), replace=False)) if paths else []

GATE_PHOTO_CLASSES = ["airplane", "car", "cat", "dog"]   # the other natural-image classes are held out
natural = find_images("natural_images/*/*.jpg")
photos_train = load_images(sample([p for p in natural if Path(p).parent.name in GATE_PHOTO_CLASSES], 600))
photos_heldout = load_images(sample([p for p in natural if Path(p).parent.name not in GATE_PHOTO_CLASSES], 300))
xray_train = load_images(sample(find_images("chest_xray/train/*/*.jpeg"), 400))
xray_heldout = load_images(sample(find_images("chest_xray/test/*/*.jpeg"), 300))
mri = load_images(sample(find_images("brain_tumor_dataset/*/*"), 250))
print({"photos (train)": len(photos_train), "photos (held out)": len(photos_heldout), "x-rays (train)": len(xray_train),
       "x-rays (held out)": len(xray_heldout), "brain MRI (never seen)": len(mri)})

def embeddings(images):
    return run_onnx(np.stack([to_gray224(im) for im in images]))[1] if images else np.zeros((0, 2048), np.float32)

negatives = np.concatenate([embeddings(photos_train), embeddings(xray_train)])
X_gate = np.concatenate([onnx_out["train"][1], negatives])
y_gate = np.r_[np.ones(len(idx["train"])), np.zeros(len(negatives))]
gate = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000, class_weight="balanced")).fit(X_gate, y_gate)
gate_score = lambda emb: gate.predict_proba(emb)[:, 1] if len(emb) else np.zeros(0)
GATE_THRESHOLD = float(min(np.percentile(gate_score(onnx_out["val"][1]), 1), 0.5))

test_colour = np.array([colour_fraction(Image.open(p)) for p in df["path"].values[idx["test"]]])
photo_scores = gate_score(embeddings(photos_heldout))   # the model only ever sees the grayscale version
eval_sets = [
    # name, colour fractions (None = already grayscale, so the colour check can't help), gate scores
    ("Test ultrasounds (should pass)", test_colour, gate_score(onnx_out["test"][1])),
    ("Colour photos, unseen categories", np.array([colour_fraction(im) for im in photos_heldout]), photo_scores),
    ("Grayscale photos, unseen categories", None, photo_scores),
    ("Chest X-rays, held out", np.array([colour_fraction(im) for im in xray_heldout]), gate_score(embeddings(xray_heldout))),
    ("Brain MRI, never seen", np.array([colour_fraction(im) for im in mri]), gate_score(embeddings(mri))),
]
ood_rows, gate_scores = [], {}
for name, colour, score in eval_sets:
    if len(score) == 0:
        print(f"{name}: dataset not attached, skipped")
        continue
    by_colour = colour > COLOUR_LIMIT if colour is not None else np.zeros(len(score), bool)
    by_gate = score < GATE_THRESHOLD
    gate_scores[name] = score
    ood_rows.append({"images": name, "n": len(score), "rejected_colour": round(float(by_colour.mean()), 4),
                     "rejected_gate": round(float(by_gate.mean()), 4),
                     "rejected_total": round(float((by_colour | by_gate).mean()), 4)})
ood_table = pd.DataFrame(ood_rows)
print(f"gate threshold {GATE_THRESHOLD:.3f}")
display(ood_table)

fig, ax = plt.subplots(figsize=(9, 4))
for name, score in gate_scores.items():
    ax.hist(score, bins=40, range=(0, 1), alpha=0.5, label=name, density=True)
ax.axvline(GATE_THRESHOLD, color="k", ls="--", label="threshold")
ax.set(xlabel="gate probability of 'breast ultrasound'", title="Ultrasound gate", yscale="log"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/breast_gate.png", dpi=130); plt.show()

## 12. Export for the Femora backend

In [ ]:
scaler, logreg = gate.named_steps["standardscaler"], gate.named_steps["logisticregression"]
np.savez(f"{OUT}/breast_gate.npz", mean=scaler.mean_.astype(np.float32), scale=scaler.scale_.astype(np.float32),
         coef=logreg.coef_[0].astype(np.float32), intercept=np.float32(logreg.intercept_[0]))

# Demo scans for the app: held-out BrEaST test images (CC BY 4.0) the model classifies correctly, median confidence
os.makedirs(f"{OUT}/samples", exist_ok=True)
test_df = df.iloc[idx["test"]].assign(pred=pred_test, conf=p_test.max(1))
samples = {}
for label in ["benign", "malignant"]:
    ok_rows = test_df[(test_df.source == "BrEaST") & (test_df.label == label) & (test_df.pred == CLASSES.index(label))]
    if len(ok_rows):
        r = ok_rows.sort_values("conf").iloc[len(ok_rows) // 2]
        shutil.copy(r["path"], f"{OUT}/samples/breast_sample_{label}.png")
        samples[label] = os.path.basename(r["path"])

df.assign(file=df["path"].map(os.path.basename))[["file", "source", "label", "group", "split", "case", "device"]] \
  .to_csv(f"{OUT}/breast_split.csv", index=False)

def to_builtin(o):
    return o.item() if hasattr(o, "item") else str(o)

with open(f"{OUT}/breast_meta.json", "w") as f:
    json.dump({
        "version": "busbra (v7 pipeline + BUS-BRA)",
        "classes": CLASSES,
        "onnx_file": "breast_resnet50.onnx",
        "outputs": ["logits", "embedding", "cam"],
        "input": {"size": IMG, "mean": MEAN.ravel().tolist(), "std": STD.ravel().tolist(),
                  "preprocessing": "EXIF-rotate, grayscale, pad to square with black, bilinear resize to 224, "
                                   "repeat to 3 channels, ImageNet normalisation"},
        "temperature": TEMPERATURE,
        "malignant_threshold": MAL_THRESHOLD,
        "colour_limit": COLOUR_LIMIT,
        "gate_file": "breast_gate.npz",
        "gate_threshold": GATE_THRESHOLD,
        "recipe": {"name": BEST_RECIPE, **RECIPES[BEST_RECIPE]},
        "recipe_comparison": recipe_table.to_dict(orient="records"),
        "test_metrics": test_metrics,
        "test_metrics_argmax": argmax_metrics,
        "per_source_test": per_source,
        "test_metrics_v7_subset": test_v7_subset,
        "test_metrics_busbra": test_busbra,
        "per_device_test": per_device,
        "comparison": comparison.to_dict(orient="records"),
        "cv_results": cv_table.to_dict(orient="records"),
        "cv_summary": cv_summary,
        "calibration": calibration,
        "cam_evaluation": cam_eval,
        "ood_evaluation": ood_table.to_dict(orient="records"),
        "onnx_parity": {"fp32_export_max_abs_diff": parity_fp32, "deployed_max_abs_diff": parity,
                        "test_predictions_changed_by_fp16": flipped},
        "best_epoch": best_epoch,
        "samples": samples,
        "dataset": {
            "sources": ["BUSI (Al-Dhabyani et al., 2020)", "BrEaST-Lesions-USG (Pawłowska et al., 2024, TCIA, CC BY 4.0)",
                    "BUS-BRA (Gómez-Flores et al., Medical Physics 2024)"],
            "images": int(len(df)), "near_duplicates": dedup_stats,
            "counts": pd.crosstab(df["source"], df["label"]).to_dict(),
            "splits": df["split"].value_counts().to_dict(),
        },
    }, f, indent=2, default=to_builtin)
print(sorted(os.listdir(OUT)))